# WorkBot Micro-Specialist: fine-tune a 1.5B / 3B "Muscle" model for tool calling and browser actions

<a href="https://colab.research.google.com/github/AaronGrace978/The-Ai/blob/main/WorkBot_Micro_Specialist_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook turns **Qwen2.5-Coder 1.5B or 3B Instruct** into a single-purpose execution model:

* **Tool caller.** Given a tool list and a request, it answers with `<tool_call>{...}</tool_call>` JSON right away, with no chatty preamble.
* **Browser actor (DOM to action).** Given a task, the previous actions and a list of interactive page elements, it returns one `browser_act` call (`CLICK` / `TYPE` / `SELECT` plus an element id). This is the payload a Tauri/Rust layer can execute directly.
* **Knows when *not* to call.** It asks for missing arguments or declines when no tool fits, instead of making up calls.

Everything runs on **free tooling**: Colab's free T4 GPU (or your own RTX 5060 Ti 16 GB), Unsloth with 4-bit QLoRA, open Hugging Face datasets, and an export to **GGUF + Ollama** so the model runs locally and offline.

### Training data (all open; mixed and converted to Qwen's native tool format)

| Source | What it teaches |
|---|---|
| [NousResearch/hermes-function-calling-v1](https://huggingface.co/datasets/NousResearch/hermes-function-calling-v1) (single-turn, multi-turn, JSON-mode agentic) | Clean single and parallel tool calls, using tool results across turns, schema-following JSON output |
| [Team-ACE/ToolACE](https://huggingface.co/datasets/Team-ACE/ToolACE) | 26k+ diverse API schemas, nested arguments, multi-turn tool use |
| [Salesforce/xlam-function-calling-60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) *(optional, gated: accept the license on the Hub and add `HF_TOKEN`)* | Verified, executable function calls (the data behind the xLAM models) |
| [nvidia/When2Call](https://huggingface.co/datasets/nvidia/When2Call) | Asking for missing parameters and refusing when no tool fits (reduces hallucinated calls) |
| [osunlp/Mind2Web](https://huggingface.co/datasets/osunlp/Mind2Web) | Real human browser trajectories on 100+ websites, converted to *element list → `browser_act`* steps |
| [ise-uiuc/Magicoder-OSS-Instruct-75K](https://huggingface.co/datasets/ise-uiuc/Magicoder-OSS-Instruct-75K) | A small slice of coding instructions, so the coder base keeps its coding skill |
| **Your own WorkBot trajectories** *(optional JSONL)* | Your exact workflows, oversampled so they dominate |

### Honest expectations

A 1.5B/3B specialist will **not** replace Claude, GPT or Cursor for open-ended reasoning or large refactors. It is a fast, cheap, local *muscle*: structured tool calls, DOM-to-action translation, form filling, routing. Keep a bigger model (cloud or a local 7B/14B) as the *mind* for planning, and let this model handle the high-volume execution steps. The before/after evaluation in this notebook measures how well it does that job on held-out data.

### Runtime guide

| Preset | Base model | Hardware | Approx. time |
|---|---|---|---|
| `colab_free` | Qwen2.5-Coder-1.5B-Instruct | Colab free T4 | about 1 to 1.5 h of training |
| `quality` | Qwen2.5-Coder-3B-Instruct | Colab L4/A100, or a local RTX 5060 Ti 16 GB | several hours (good overnight run) |

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better). Then run the cells top to bottom.

## 1. Check the GPU and install Unsloth

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Unsloth pulls in compatible torch / transformers / trl / peft / bitsandbytes versions.
# Running this notebook locally on an RTX 50-series (Blackwell) card needs a CUDA 12.8+ build of PyTorch.
!pip install -q unsloth
!pip install -q lxml pyarrow pandas

## 2. Configuration

Pick a preset, then optionally:

* set `WORKBOT_DATA_PATH` to a JSONL file of your own trajectories (format in section 5), and
* set `HF_REPO` to push the finished model to your Hugging Face account.

Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) to unlock the gated xLAM dataset and pushing to the Hub. Without it the notebook still runs, but skips xLAM.

In [ ]:
import os, json, random, re, math, glob, ast, time
from collections import Counter, defaultdict

PRESET = "colab_free"   # "colab_free" (free T4)  |  "quality" (3B; L4/A100 or a local RTX 5060 Ti)

PRESETS = {
    "colab_free": dict(
        base_model="unsloth/Qwen2.5-Coder-1.5B-Instruct",
        max_seq_length=4096,
        lora_r=32,
        epochs=1,
        batch_size=2,
        grad_accum=4,
        learning_rate=2e-4,
        # Number of training examples drawn from each source (None = use everything available).
        mix=dict(hermes_single=800, hermes_multi=600, hermes_json=300, toolace=1500,
                 xlam=1000, when2call=500, mind2web=1500, magicoder=300),
        mind2web_shards=3,      # each shard is about 90-110 MB and holds about 100-125 tasks (about 750-900 steps)
        eval_per_set=40,
    ),
    "quality": dict(
        base_model="unsloth/Qwen2.5-Coder-3B-Instruct",
        max_seq_length=6144,
        lora_r=64,
        epochs=1,
        batch_size=2,
        grad_accum=8,
        learning_rate=1.5e-4,
        mix=dict(hermes_single=None, hermes_multi=None, hermes_json=None, toolace=8000,
                 xlam=10000, when2call=3000, mind2web=None, magicoder=3000),
        mind2web_shards=8,
        eval_per_set=100,
    ),
}
CFG = PRESETS[PRESET]

WORKBOT_DATA_PATH = ""    # e.g. "/content/drive/MyDrive/workbot/trajectories.jsonl"
WORKBOT_OVERSAMPLE = 3    # repeat your own examples so they outweigh the public data

OUTPUT_DIR = "workbot_muscle_run"   # point at Google Drive to survive Colab disconnects
EXPORT_NAME = "workbot-muscle-" + ("1.5b" if "1.5B" in CFG["base_model"] else "3b")
GGUF_QUANT = "q4_k_m"     # q4_k_m is about 1 GB for 1.5B and 2 GB for 3B; use "q8_0" for max quality
HF_REPO = ""              # e.g. "your-username/workbot-muscle-1.5b" to push LoRA + GGUF to the Hub

SEED = 3407
random.seed(SEED)

WORKBOT_SYSTEM = (
    "You are WorkBot Muscle, a fast execution agent. "
    "When a provided tool can do the job, respond with tool calls only and no other text. "
    "If a required argument is missing, ask one short question for it. "
    "If no tool fits the request, say so in one sentence."
)

BROWSER_TOOL = {
    "type": "function",
    "function": {
        "name": "browser_act",
        "description": "Perform exactly one action on the current web page. "
                       "element_id must be one of the [id] values from the ELEMENTS list.",
        "parameters": {
            "type": "object",
            "properties": {
                "action": {"type": "string", "enum": ["CLICK", "TYPE", "SELECT"],
                           "description": "CLICK an element, TYPE text into it, or SELECT an option"},
                "element_id": {"type": "string", "description": "id of the target element"},
                "value": {"type": "string", "description": "text to type or option to select; omit for CLICK"},
            },
            "required": ["action", "element_id"],
        },
    },
}
print("Preset:", PRESET, "->", CFG["base_model"])

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN found: the gated xLAM dataset and Hub upload will be skipped.")

# Optional: keep checkpoints and exports on Google Drive.
# from google.colab import drive; drive.mount("/content/drive")
# OUTPUT_DIR = "/content/drive/MyDrive/workbot_muscle_run"

## 3. Load the base model in 4-bit and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["base_model"],
    max_seq_length=CFG["max_seq_length"],
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG["lora_r"],
    lora_alpha=CFG["lora_r"],
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

## 4. Build the training mix

Every source is converted into one shape: OpenAI-style `messages` (`system`, `user`, `assistant` with `tool_calls`, `tool`) plus a `tools` list in JSON-Schema form. It is then rendered with **Qwen's own chat template**, so the fine-tuned model speaks exactly the format that Ollama, llama.cpp and vLLM expect for Qwen tool calling.

In [ ]:
TYPE_MAP = {"dict": "object", "str": "string", "string": "string", "int": "integer", "integer": "integer",
            "float": "number", "number": "number", "bool": "boolean", "boolean": "boolean",
            "list": "array", "array": "array", "tuple": "array", "object": "object", "any": "string"}

def _norm_type(t):
    if not isinstance(t, str):
        return "string"
    base = t.split(",")[0].strip().lower()
    base = re.sub(r"\[.*", "", base)
    return TYPE_MAP.get(base, "string")

def _norm_schema(s):
    if not isinstance(s, dict):
        return {"type": "string"}
    out = dict(s)
    out["type"] = _norm_type(s.get("type", "object" if "properties" in s else "string"))
    if isinstance(s.get("properties"), dict):
        out["properties"] = {k: _norm_schema(v) for k, v in s["properties"].items()}
    if isinstance(s.get("items"), dict):
        out["items"] = _norm_schema(s["items"])
    return out

def norm_tool(t):
    """Normalise Hermes / ToolACE / When2Call / xLAM tool specs to OpenAI JSON-Schema tools."""
    if isinstance(t, str):
        t = json.loads(t)
    fn = t.get("function", t)
    params = fn.get("parameters") or {"type": "object", "properties": {}}
    if "properties" not in params and params.get("type") not in ("object", "dict"):
        # xLAM style: {"arg": {"type": "str", "description": ..., "default": ...}}
        props, req = {}, []
        for k, v in params.items():
            v = v if isinstance(v, dict) else {"type": "str"}
            props[k] = v
            optional = "optional" in str(v.get("type", "")).lower() or "default" in v
            if not optional:
                req.append(k)
        params = {"type": "object", "properties": props, "required": req}
    params = _norm_schema(params)
    params["type"] = "object"
    params.setdefault("properties", {})
    req = params.get("required") or fn.get("required") or []
    params["required"] = [r for r in req if r in params["properties"]] if isinstance(req, list) else []
    return {"type": "function",
            "function": {"name": str(fn["name"]), "description": str(fn.get("description", "")), "parameters": params}}

def tool_call(name, args):
    if isinstance(args, str):
        args = json.loads(args) if args.strip() else {}
    return {"type": "function", "function": {"name": name, "arguments": args}}

def parse_json_loose(s):
    try:
        return json.loads(s)
    except Exception:
        return ast.literal_eval(s)

TOOL_CALL_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)
TOOL_RESP_RE = re.compile(r"<tool_response>\s*(.*?)\s*</tool_response>", re.S)

def hermes_assistant(text):
    """'<tool_call>{json}</tool_call>' blocks -> assistant message with tool_calls."""
    blobs = TOOL_CALL_RE.findall(text)
    if not blobs:
        return {"role": "assistant", "content": text.strip()}
    calls = []
    for b in blobs:
        obj = parse_json_loose(b)
        calls.append(tool_call(obj["name"], obj.get("arguments", {})))
    rest = TOOL_CALL_RE.sub("", text).strip()
    return {"role": "assistant", "content": rest, "tool_calls": calls}

# ToolACE writes calls as "[Func Name(a=1, b='x'), Other(c=[1,2])]".
_PY_CONSTS = {"true": True, "false": False, "null": None, "none": None, "True": True, "False": False, "None": None}

def _lit(node):
    try:
        return ast.literal_eval(node)
    except Exception:
        if isinstance(node, ast.Name) and node.id in _PY_CONSTS:
            return _PY_CONSTS[node.id]
        if isinstance(node, ast.List):
            return [_lit(e) for e in node.elts]
        if isinstance(node, ast.Dict):
            return {_lit(k): _lit(v) for k, v in zip(node.keys, node.values)}
        return ast.unparse(node)

def _split_top(s):
    parts, depth, quote, cur = [], 0, None, ""
    for i, ch in enumerate(s):
        if quote:
            cur += ch
            if ch == quote and s[i - 1] != "\\":
                quote = None
            continue
        if ch in "\"'":
            quote = ch
        elif ch in "([{":
            depth += 1
        elif ch in ")]}":
            depth -= 1
        if ch == "," and depth == 0:
            parts.append(cur)
            cur = ""
        else:
            cur += ch
    if cur.strip():
        parts.append(cur)
    return [p.strip() for p in parts]

PY_CALLS_RE = re.compile(r"^\[\s*[^\[\]()]+\(.*\)\s*\]$", re.S)

def parse_python_calls(text):
    text = text.strip()
    if not PY_CALLS_RE.match(text):
        return None
    calls = []
    for part in _split_top(text[1:-1]):
        name, _, rest = part.partition("(")
        if not rest.endswith(")"):
            return None
        node = ast.parse("f(" + rest[:-1] + ")", mode="eval").body
        if node.args:
            return None
        calls.append(tool_call(name.strip(), {k.arg: _lit(k.value) for k in node.keywords}))
    return calls

In [ ]:
from datasets import load_dataset

def take(ds, n):
    ds = ds.shuffle(seed=SEED)
    return ds if n is None else ds.select(range(min(n, len(ds))))

def full(n):
    return lambda out: n is not None and len(out) >= n

def load_hermes(config, n, tag):
    ds = take(load_dataset("NousResearch/hermes-function-calling-v1", config, split="train"), None)
    done = full(n)
    out, bad = [], 0
    for row in ds:
        if done(out):
            break
        try:
            tools = [norm_tool(t) for t in json.loads(row["tools"])] if row.get("tools") else None
            msgs = []
            for turn in row["conversations"]:
                role, val = turn["from"], turn["value"]
                if role == "system":
                    # JSON-mode rows keep their schema prompt; tool rows use the WorkBot prompt.
                    msgs.append({"role": "system", "content": WORKBOT_SYSTEM if tools else val})
                elif role == "human":
                    msgs.append({"role": "user", "content": val})
                elif role == "gpt":
                    msgs.append(hermes_assistant(val))
                elif role == "tool":
                    for resp in (TOOL_RESP_RE.findall(val) or [val]):
                        msgs.append({"role": "tool", "content": resp.strip()})
            if msgs[0]["role"] != "system":
                msgs.insert(0, {"role": "system", "content": WORKBOT_SYSTEM})
            out.append({"source": tag, "messages": msgs, "tools": tools})
        except Exception:
            bad += 1
    print(f"{tag:14s} {len(out):6d} examples  ({bad} malformed rows filtered)")
    return out

TOOLACE_TOOLS_RE = re.compile(r"invoke:\s*(\[.*\])\s*\.\s*\n\s*Should you decide", re.S)

def load_toolace(n):
    ds = take(load_dataset("Team-ACE/ToolACE", split="train"), None)
    done = full(n)
    out, bad = [], 0
    for row in ds:
        if done(out):
            break
        try:
            m = TOOLACE_TOOLS_RE.search(row["system"])
            tools = [norm_tool(t) for t in json.loads(m.group(1))]
            msgs = [{"role": "system", "content": WORKBOT_SYSTEM}]
            for turn in row["conversations"]:
                role, val = turn["from"], turn["value"]
                if role == "user":
                    msgs.append({"role": "user", "content": val})
                elif role == "assistant":
                    calls = parse_python_calls(val)
                    msgs.append({"role": "assistant", "content": "", "tool_calls": calls} if calls
                                else {"role": "assistant", "content": val})
                elif role == "tool":
                    msgs.append({"role": "tool", "content": val})
            out.append({"source": "toolace", "messages": msgs, "tools": tools})
        except Exception:
            bad += 1
    print(f"{'toolace':14s} {len(out):6d} examples  ({bad} malformed rows filtered)")
    return out

def load_when2call(n):
    ds = take(load_dataset("nvidia/When2Call", "train_sft", split="train"), n)
    out = []
    for row in ds:
        tools = [norm_tool(t) for t in row["tools"]]
        msgs = [{"role": "system", "content": WORKBOT_SYSTEM}] + [
            {"role": m["role"], "content": m["content"]} for m in row["messages"]]
        out.append({"source": "when2call", "messages": msgs, "tools": tools})
    print(f"{'when2call':14s} {len(out):6d} examples")
    return out

def load_xlam(n):
    try:
        ds = take(load_dataset("Salesforce/xlam-function-calling-60k", split="train", token=HF_TOKEN), None)
    except Exception as e:
        print(f"{'xlam':14s} skipped (gated: accept the license on the Hub and set HF_TOKEN). {type(e).__name__}")
        return []
    done = full(n)
    out, bad = [], 0
    for row in ds:
        if done(out):
            break
        try:
            tools = [norm_tool(t) for t in json.loads(row["tools"])]
            calls = [tool_call(a["name"], a.get("arguments", {})) for a in json.loads(row["answers"])]
            msgs = [{"role": "system", "content": WORKBOT_SYSTEM},
                    {"role": "user", "content": row["query"]},
                    {"role": "assistant", "content": "", "tool_calls": calls}]
            out.append({"source": "xlam", "messages": msgs, "tools": tools})
        except Exception:
            bad += 1
    print(f"{'xlam':14s} {len(out):6d} examples  ({bad} malformed rows filtered)")
    return out

def load_magicoder(n):
    ds = take(load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train"), n)
    out = [{"source": "magicoder", "tools": None, "messages": [
        {"role": "user", "content": r["problem"]}, {"role": "assistant", "content": r["solution"]}]} for r in ds]
    print(f"{'magicoder':14s} {len(out):6d} examples")
    return out

### Mind2Web → `browser_act` steps

Each human action in Mind2Web becomes one training step. The prompt contains the task, the actions taken so far and a compact list of interactive elements (the correct target plus up to 24 real distractors from the same page, in document order). The answer is a single `browser_act` tool call.

This prompt layout is the contract for your Tauri/Rust layer: strip the live DOM to interactive elements, print them in this format, and execute the returned call.

```
TASK: Check for pickup restaurant available in Boston, NY on March 18, 5pm with just one guest
WEBSITE: exploretock
PREVIOUS ACTIONS:
1. [combobox]  Reservation type -> SELECT: Pickup
2. [svg]   -> CLICK
ELEMENTS:
[6541] a "Skip to main content"
[6068] input type=search placeholder="Find a location" value="Columbus, OH, US"
...
```

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from lxml import html as lxml_html

INTERACTIVE_TAGS = {"a", "button", "input", "select", "textarea", "option", "label", "img", "svg", "li", "span", "td", "th"}
SKIP_ATTRS = {"backend_node_id", "bounding_box_rect", "class", "style", "data_pw_testid_buckeye_candidate", "is_clickable"}
MAX_DISTRACTORS = 24

def _clip(s, n):
    s = re.sub(r"\s+", " ", s or "").strip()
    return s if len(s) <= n else s[: n - 1] + "…"

def render_element(node_id, el):
    attrs = []
    for k, v in el.attrib.items():
        if k in SKIP_ATTRS or not v or len(attrs) >= 4:
            continue
        attrs.append(f'{k}="{_clip(v, 40)}"' if " " in v or k in ("placeholder", "value", "title", "aria_label") else f"{k}={_clip(v, 40)}")
    text = _clip(el.text_content(), 80)
    line = f"[{node_id}] {el.tag}"
    if attrs:
        line += " " + " ".join(attrs)
    if text:
        line += f' "{text}"'
    return line

def mind2web_step(task, website, history, action, rng):
    root = lxml_html.fromstring(action["cleaned_html"])
    order = {}
    by_id = {}
    for i, el in enumerate(root.iter()):
        nid = el.get("backend_node_id")
        if nid is not None:
            order[nid] = i
            by_id[nid] = el
    pos = [c for c in action["pos_candidates"] if c["backend_node_id"] in by_id]
    if not pos:
        return None
    target = pos[0]["backend_node_id"]
    negs = []
    for c in action["neg_candidates"]:
        el = by_id.get(c["backend_node_id"])
        if el is None:
            continue
        text_len = len(re.sub(r"\s+", " ", el.text_content()).strip())
        clickable = "is_clickable" in (c.get("attributes") or "") or el.get("role") is not None
        if (el.tag in INTERACTIVE_TAGS or clickable) and text_len <= 120:
            negs.append(c["backend_node_id"])
    rng.shuffle(negs)
    ids = sorted(set([target] + negs[:MAX_DISTRACTORS]), key=lambda x: order[x])
    elements = "\n".join(render_element(i, by_id[i]) for i in ids)
    hist = "\n".join(f"{k + 1}. {h}" for k, h in enumerate(history[-8:])) or "(none)"
    op = action["operation"]["op"]
    args = {"action": op, "element_id": target}
    if op in ("TYPE", "SELECT"):
        args["value"] = action["operation"]["value"]
    user = f"TASK: {task}\nWEBSITE: {website}\nPREVIOUS ACTIONS:\n{hist}\nELEMENTS:\n{elements}"
    return [{"role": "system", "content": WORKBOT_SYSTEM},
            {"role": "user", "content": user},
            {"role": "assistant", "content": "", "tool_calls": [tool_call("browser_act", args)]}]

def load_mind2web(n_shards):
    """Returns {task_id: [examples]} so train/eval can be split by task (no page leakage)."""
    rng = random.Random(SEED)
    tasks = {}
    for shard in range(n_shards):
        try:
            path = hf_hub_download("osunlp/Mind2Web", f"default/partial-train/{shard:04d}.parquet",
                                   repo_type="dataset", revision="refs/convert/parquet")
        except Exception as e:
            print(f"mind2web shard {shard} unavailable: {type(e).__name__}")
            break
        pf = pq.ParquetFile(path)
        for rg in range(pf.num_row_groups):
            table = pf.read_row_group(rg, columns=["annotation_id", "website", "confirmed_task", "action_reprs", "actions"])
            for b in range(0, table.num_rows, 8):
                for row in table.slice(b, 8).to_pylist():
                    steps = []
                    for i, act in enumerate(row["actions"]):
                        try:
                            msgs = mind2web_step(row["confirmed_task"], row["website"], row["action_reprs"][:i], act, rng)
                        except Exception:
                            msgs = None
                        if msgs:
                            steps.append({"source": "mind2web", "messages": msgs, "tools": [BROWSER_TOOL]})
                    if steps:
                        tasks[row["annotation_id"]] = steps
            del table
    print(f"{'mind2web':14s} {sum(map(len, tasks.values())):6d} steps from {len(tasks)} tasks")
    return tasks

## 5. (Optional) Your own WorkBot trajectories

This is where the model becomes *yours*. Export the successful runs from your WorkBot loop (the "Gold Standard" playbooks) as JSONL, one example per line, in either of these shapes:

**A. Full chat with tools** (OpenAI format; `arguments` may be an object or a JSON string):
```json
{"tools": [{"type": "function", "function": {"name": "open_repo", "description": "...", "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}}],
 "messages": [{"role": "user", "content": "open the tauri app repo"},
              {"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": "open_repo", "arguments": {"path": "~/code/workbot-tauri"}}}]}]}
```

**B. One browser step** (same layout as the Mind2Web prompt above):
```json
{"task": "Log in to the dashboard", "website": "myapp.local", "history": ["[input] Email -> TYPE: me@x.com"],
 "elements": "[12] input type=password placeholder=\"Password\"\n[13] button \"Sign in\"",
 "action": {"action": "CLICK", "element_id": "13"}}
```

A few hundred high-quality examples of your real workflows typically matter more than thousands of generic ones. They are repeated `WORKBOT_OVERSAMPLE` times in the mix.

In [ ]:
def load_workbot(path, oversample):
    if not path:
        print(f"{'workbot':14s} no file set (WORKBOT_DATA_PATH is empty)")
        return []
    out, bad = [], 0
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            try:
                row = json.loads(line)
                if "messages" in row:
                    msgs = [m for m in row["messages"]]
                    if msgs[0]["role"] != "system":
                        msgs.insert(0, {"role": "system", "content": WORKBOT_SYSTEM})
                    for m in msgs:
                        if m.get("tool_calls"):
                            m["tool_calls"] = [tool_call(c["function"]["name"], c["function"].get("arguments", {}))
                                               for c in m["tool_calls"]]
                            m.setdefault("content", "")
                    tools = [norm_tool(t) for t in row.get("tools") or []] or None
                else:
                    hist = "\n".join(f"{k + 1}. {h}" for k, h in enumerate(row.get("history", [])[-8:])) or "(none)"
                    user = (f"TASK: {row['task']}\nWEBSITE: {row.get('website', '')}\n"
                            f"PREVIOUS ACTIONS:\n{hist}\nELEMENTS:\n{row['elements']}")
                    args = {k: v for k, v in row["action"].items() if v not in (None, "")}
                    msgs = [{"role": "system", "content": WORKBOT_SYSTEM}, {"role": "user", "content": user},
                            {"role": "assistant", "content": "", "tool_calls": [tool_call("browser_act", args)]}]
                    tools = [BROWSER_TOOL]
                out.append({"source": "workbot", "messages": msgs, "tools": tools})
            except Exception:
                bad += 1
    print(f"{'workbot':14s} {len(out):6d} examples x{oversample}  ({bad} skipped)")
    return out * oversample

## 6. Assemble, render with the chat template, and hold out evaluation sets

In [ ]:
mix = CFG["mix"]
E = CFG["eval_per_set"]

def split_off(records, n_eval):
    random.Random(SEED).shuffle(records)
    return records[n_eval:], records[:n_eval]

pools = {}
pools["hermes_single"] = load_hermes("func_calling_singleturn", None if mix["hermes_single"] is None else mix["hermes_single"] + E, "hermes_single")
pools["hermes_multi"] = load_hermes("func_calling", mix["hermes_multi"], "hermes_multi")
pools["hermes_json"] = load_hermes("json_mode_agentic", mix["hermes_json"], "hermes_json")
pools["toolace"] = load_toolace(None if mix["toolace"] is None else mix["toolace"] + E)
pools["xlam"] = load_xlam(None if mix["xlam"] is None else mix["xlam"] + E)
pools["when2call"] = load_when2call(None if mix["when2call"] is None else mix["when2call"] + E)
pools["magicoder"] = load_magicoder(mix["magicoder"])

m2w_tasks = load_mind2web(CFG["mind2web_shards"])
task_ids = sorted(m2w_tasks)
random.Random(SEED).shuffle(task_ids)
eval_m2w, train_m2w, n_eval_tasks = [], [], 0
for tid in task_ids:
    if len(eval_m2w) < E:
        eval_m2w += m2w_tasks[tid]
        n_eval_tasks += 1
    else:
        train_m2w += m2w_tasks[tid]
random.Random(SEED).shuffle(train_m2w)
train_records = train_m2w[: mix["mind2web"]] if mix["mind2web"] else train_m2w
eval_sets = {"mind2web": eval_m2w[:E]}
print(f"mind2web eval uses {n_eval_tasks} unseen tasks")

for name in ["hermes_single", "toolace", "xlam", "when2call"]:
    if pools[name]:
        pools[name], eval_sets[name] = split_off(pools[name], E)
for name, recs in pools.items():
    train_records += recs[: mix[name]] if mix[name] else recs
train_records += load_workbot(WORKBOT_DATA_PATH, WORKBOT_OVERSAMPLE)
random.Random(SEED).shuffle(train_records)

In [ ]:
from datasets import Dataset

def render(rec, add_generation_prompt=False):
    return tokenizer.apply_chat_template(rec["messages"], tools=rec["tools"] or None, tokenize=False,
                                         add_generation_prompt=add_generation_prompt)

rows, too_long, failed = [], 0, 0
for rec in train_records:
    try:
        text = render(rec)
    except Exception:
        failed += 1
        continue
    if len(tokenizer(text, add_special_tokens=False)["input_ids"]) > CFG["max_seq_length"]:
        too_long += 1
        continue
    rows.append({"text": text, "source": rec["source"]})

train_ds = Dataset.from_list(rows)
print(f"Training examples: {len(train_ds)}   (dropped {too_long} over {CFG['max_seq_length']} tokens, {failed} failed to render)")
print(Counter(train_ds["source"]).most_common())
print("\n========== sample: browser step ==========")
print(next(r["text"] for r in rows if r["source"] == "mind2web")[-1500:])

## 7. Evaluation harness (run before and after training)

For every held-out example the model sees the conversation up to the first assistant turn and generates greedily. Metrics:

* **decision**: called a tool when it should have, and stayed quiet (asked or declined) when it should not have
* **valid_json**: every `<tool_call>` block parses as JSON
* **name**: the set of called function names is exactly right
* **exact**: names *and* arguments match (values compared case-insensitively as strings)
* **element** (Mind2Web): the right element id was chosen

In [ ]:
def eval_item(rec):
    msgs = rec["messages"]
    i = next(k for k, m in enumerate(msgs) if m["role"] == "assistant")
    target = msgs[i]
    expected = [(c["function"]["name"], c["function"]["arguments"]) for c in target.get("tool_calls") or []]
    prompt = tokenizer.apply_chat_template(msgs[:i], tools=rec["tools"] or None, tokenize=False, add_generation_prompt=True)
    return prompt, expected

def parse_calls(text):
    calls, valid = [], True
    for blob in TOOL_CALL_RE.findall(text):
        try:
            obj = json.loads(blob)
            calls.append((obj["name"], obj.get("arguments", {})))
        except Exception:
            valid = False
    if "<tool_call>" in text and not calls:
        valid = False
    return calls, valid

def canon(v):
    if isinstance(v, dict):
        return tuple(sorted((str(k), canon(x)) for k, x in v.items()))
    if isinstance(v, (list, tuple)):
        return tuple(canon(x) for x in v)
    if isinstance(v, str):
        try:
            parsed = json.loads(v)
            if isinstance(parsed, (dict, list)):
                return canon(parsed)
        except Exception:
            pass
    return str(v).strip().lower()

def score(output, expected):
    calls, valid = parse_calls(output)
    should_call = bool(expected)
    s = {"decision": float(bool(calls) == should_call), "valid_json": float(valid)}
    if should_call:
        s["name"] = float(sorted(n for n, _ in calls) == sorted(n for n, _ in expected))
        s["exact"] = float(sorted(map(canon, calls)) == sorted(map(canon, expected)))
        if expected[0][0] == "browser_act":
            got = calls[0][1].get("element_id") if calls and isinstance(calls[0][1], dict) else None
            s["element"] = float(str(got) == str(expected[0][1]["element_id"]))
    else:
        s["exact"] = s["decision"]
    return s

In [ ]:
import pandas as pd

@torch.no_grad()
def generate(prompts, max_new_tokens=384, batch_size=8):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    outs = []
    for b in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[b:b + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                             pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tokenizer.padding_side = "right"
    return outs

def evaluate(label):
    table, examples = {}, {}
    for name, recs in eval_sets.items():
        items = [eval_item(r) for r in recs]
        outputs = generate([p for p, _ in items])
        scores = [score(o, e) for o, (_, e) in zip(outputs, items)]
        keys = sorted({k for s in scores for k in s})
        table[name] = {k: round(100 * sum(s[k] for s in scores if k in s) / max(1, sum(k in s for s in scores)), 1) for k in keys}
        table[name]["n"] = len(scores)
        examples[name] = (items[0][1], outputs[0])
    df = pd.DataFrame(table).T
    print(f"\n===== {label} (% correct) =====")
    print(df.to_string())
    return df, examples

t0 = time.time()
baseline_df, baseline_examples = evaluate("BASE MODEL, before fine-tuning")
print(f"(eval took {time.time() - t0:.0f}s)")

## 8. Train

Only the assistant turns are trained (`train_on_responses_only`), so the model learns to *produce* calls and answers, not to imitate tool outputs or user text. Checkpoints are written every 200 steps; if Colab disconnects, re-run the notebook and training resumes from the last checkpoint in `OUTPUT_DIR`.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)
tokenizer.padding_side = "right"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=CFG["batch_size"],
        gradient_accumulation_steps=CFG["grad_accum"],
        num_train_epochs=CFG["epochs"],
        learning_rate=CFG["learning_rate"],
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        save_steps=200,
        save_total_limit=2,
        output_dir=OUTPUT_DIR,
        seed=SEED,
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

steps = math.ceil(len(train_ds) / (CFG["batch_size"] * CFG["grad_accum"])) * CFG["epochs"]
print(f"{len(train_ds)} examples -> about {steps} optimizer steps")
resume = bool(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))
stats = trainer.train(resume_from_checkpoint=resume)
print(f"Done in {stats.metrics['train_runtime'] / 60:.1f} min, final loss {stats.metrics['train_loss']:.4f}")

## 9. Evaluate the fine-tuned model

In [ ]:
tuned_df, tuned_examples = evaluate("FINE-TUNED")
compare = pd.concat({"before": baseline_df.drop(columns="n"), "after": tuned_df.drop(columns="n")}, axis=1)
print("\n===== before vs after =====")
print(compare.to_string())
for name, (expected, out) in tuned_examples.items():
    print(f"\n--- {name}: expected {expected}\n{out[:400]}")

## 10. Try it: a live browser step and a normal tool call

In [ ]:
def ask(messages, tools, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)
    t0 = time.time()
    out = generate([prompt], max_new_tokens=max_new_tokens)[0]
    print(f"[{time.time() - t0:.2f}s]\n{out}\n")
    return out

page = """TASK: Search for wireless headphones under $100 and sort by customer rating
WEBSITE: shop.example.com
PREVIOUS ACTIONS:
1. [input]  Search -> TYPE: wireless headphones
2. [button]  Search -> CLICK
ELEMENTS:
[101] a "Today's Deals"
[214] input type=checkbox aria_label="Under $100"
[215] span "Under $100"
[230] select name=sort "Featured Price: Low to High Avg. Customer Review"
[301] a "Sony WH-CH720N Wireless Headphones"
[388] button "Add to Cart"
"""
ask([{"role": "system", "content": WORKBOT_SYSTEM}, {"role": "user", "content": page}], [BROWSER_TOOL])

dev_tools = [norm_tool({"name": "run_tests", "description": "Run the test suite of a repository",
                        "parameters": {"type": "object", "properties": {
                            "repo": {"type": "string"}, "pattern": {"type": "string", "description": "test name filter"}},
                            "required": ["repo"]}}),
             norm_tool({"name": "git_commit", "description": "Commit staged changes",
                        "parameters": {"type": "object", "properties": {"repo": {"type": "string"}, "message": {"type": "string"}},
                                       "required": ["repo", "message"]}})]
ask([{"role": "system", "content": WORKBOT_SYSTEM},
     {"role": "user", "content": "Run only the auth tests in workbot-core, then commit with message 'fix token refresh'."}], dev_tools)
ask([{"role": "system", "content": WORKBOT_SYSTEM}, {"role": "user", "content": "Commit my changes."}], dev_tools)

## 11. Export: LoRA adapters, GGUF and an Ollama Modelfile

The GGUF export compiles llama.cpp the first time (several minutes). The result is one file you can run with Ollama, LM Studio or llama.cpp on your RTX 5060 Ti.

In [ ]:
LORA_DIR = os.path.join(OUTPUT_DIR, "lora")
GGUF_DIR = os.path.join(OUTPUT_DIR, "gguf")
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print("LoRA adapters saved to", LORA_DIR)

model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=GGUF_QUANT)
ggufs = sorted(set(glob.glob(os.path.join(GGUF_DIR, "**", "*.gguf"), recursive=True)
                   + glob.glob(f"{GGUF_DIR}*/*.gguf") + glob.glob("*.gguf")), key=os.path.getmtime)
assert ggufs, "GGUF export produced no file; check the llama.cpp build log above."
gguf_path = next((g for g in reversed(ggufs) if GGUF_QUANT.lower() in g.lower()), ggufs[-1])
print("GGUF:", gguf_path, f"({os.path.getsize(gguf_path) / 1e9:.2f} GB)")

In [ ]:
# Ollama template for Qwen2.5 tool calling; it mirrors the prompt format used in training.
OLLAMA_TEMPLATE = """{{- if .Messages }}
{{- if or .System .Tools }}<|im_start|>system
{{- if .System }}
{{ .System }}
{{- end }}
{{- if .Tools }}

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{{- range .Tools }}
{"type": "function", "function": {{ .Function }}}
{{- end }}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
{{- end }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if .Content }}{{ .Content }}
{{- else if .ToolCalls }}<tool_call>
{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{ end }}</tool_call>
{{- end }}{{ if not $last }}<|im_end|>
{{ end }}
{{- else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}
{{- else }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}"""

export_dir = os.path.join(OUTPUT_DIR, EXPORT_NAME)
os.makedirs(export_dir, exist_ok=True)
final_gguf = os.path.join(export_dir, f"{EXPORT_NAME}.{GGUF_QUANT}.gguf")
os.replace(gguf_path, final_gguf)
with open(os.path.join(export_dir, "Modelfile"), "w") as f:
    f.write(f"FROM ./{os.path.basename(final_gguf)}\n")
    f.write(f'TEMPLATE """{OLLAMA_TEMPLATE}"""\n')
    f.write(f'SYSTEM """{WORKBOT_SYSTEM}"""\n')
    f.write("PARAMETER temperature 0\nPARAMETER num_ctx 8192\n")
    f.write('PARAMETER stop "<|im_end|>"\nPARAMETER stop "<|endoftext|>"\n')
with open(os.path.join(export_dir, "browser_act_tool.json"), "w") as f:
    json.dump(BROWSER_TOOL, f, indent=2)
print(os.listdir(export_dir))

In [ ]:
# Push to the Hugging Face Hub (optional), then download the folder or copy it to Drive.
if HF_REPO and HF_TOKEN:
    from huggingface_hub import HfApi
    model.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO + "-lora", token=HF_TOKEN)
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(folder_path=export_dir, repo_id=HF_REPO)
    print(f"Uploaded: https://huggingface.co/{HF_REPO}")

try:
    from google.colab import files
    import shutil
    archive = shutil.make_archive(EXPORT_NAME, "zip", export_dir)
    files.download(archive)
except ImportError:
    print("Not on Colab; files are in", export_dir)

## 12. Run it locally on your RTX 5060 Ti

Install [Ollama](https://ollama.com/download), unzip the export, and from that folder:

```bash
ollama create workbot-muscle -f Modelfile
ollama run workbot-muscle "Commit my changes."
```

Ollama serves an API on `localhost:11434`, both its native `/api/chat` and an OpenAI-compatible `/v1/chat/completions` that accepts `tools`. It returns parsed `tool_calls`, so your Rust layer never parses text:

```bash
curl -s localhost:11434/api/chat -d '{
  "model": "workbot-muscle", "stream": false,
  "tools": [ <contents of browser_act_tool.json> ],
  "messages": [{"role": "user", "content": "TASK: ...\nWEBSITE: ...\nPREVIOUS ACTIONS:\n(none)\nELEMENTS:\n[12] input ...\n[13] button \"Sign in\""}]
}' | jq '.message.tool_calls'
```

**Routing sketch for the Tauri/Rust layer (mind vs. muscle):**

```rust
// Fast tier: one DOM step -> one browser_act call from the local specialist.
let body = serde_json::json!({
    "model": "workbot-muscle",
    "stream": false,
    "tools": [browser_act_tool],               // loaded from browser_act_tool.json
    "messages": [{ "role": "user", "content": step_prompt }],  // TASK / WEBSITE / PREVIOUS ACTIONS / ELEMENTS
});
let resp: serde_json::Value = client.post("http://127.0.0.1:11434/api/chat")
    .json(&body).send().await?.json().await?;
match resp["message"]["tool_calls"].get(0) {
    Some(call) => execute_browser_act(&call["function"]["arguments"]).await?,  // CLICK / TYPE / SELECT by element id
    None => escalate_to_mind(&resp["message"]["content"]).await?,               // no call: hand back to the big model
}
```

The same model also works with any tool that speaks the OpenAI API: point the base URL at `http://localhost:11434/v1` and use the model name `workbot-muscle`.

### Making it better over time

1. **Log every WorkBot step** (prompt, the action taken, and whether it succeeded).
2. Keep the successful ones as format A/B JSONL, and especially the ones where the big "mind" model had to step in, so the muscle learns from them.
3. Re-run this notebook with `WORKBOT_DATA_PATH` set. Each round makes the specialist more fitted to your sites, repos and tools.